# MAGPIE-Lite Model Training Example
This notebook demonstrates how to use the enhanced model classes for training, evaluation, and interpretation.

In [ ]:
# 1. Setup and Imports
import sys
import os
from pathlib import Path

# Add src to path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.models_enhanced import XGBoostModel, LightGBMModel, LogisticRegressionModel
from src.data_utils import DataLoader

In [ ]:
# 2. Load and Prepare Data
data_loader = DataLoader()
X, y = data_loader.load_training_data()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data: {X_train.shape}")
print(f"Test data: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

In [ ]:
# 3. Initialize Models
models = {
    "XGBoost": XGBoostModel(),
    "LightGBM": LightGBMModel(),
    "Logistic Regression": LogisticRegressionModel()
}

In [ ]:
# 4. Train and Evaluate Models
results = {}
for name, model in models.items():
    print(f\"\\nTraining {name}...\")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Evaluate
    results[name] = model.evaluate(X_test, y_test)
    
    # Plot learning curve
    model.plot_learning_curve(X_train, y_train, cv=3)
    plt.title(f"Learning Curve - {name}")
    plt.show()
    
    # Plot feature importance
    model.plot_feature_importance()
    plt.title(f"Feature Importance - {name}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Compare Models
results_df = pd.DataFrame(results).T
print("\\nModel Comparison:")
display(results_df)

In [ ]:
# 6. Hyperparameter Tuning (Example with XGBoost)
print("\\nTuning XGBoost hyperparameters...")
xgb_model = XGBoostModel()
best_params = xgb_model.tune_hyperparameters(
    X_train, 
    y_train,
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.3]
    },
    cv=3,
    n_iter=10
)

print("\\nBest parameters found:")
for param, value in best_params.items():
    print(f"{param}: {value}")

In [ ]:
# 7. Train Final Model with Best Parameters
print("\\nTraining final model with best parameters...")
final_model = XGBoostModel(**best_params)
final_model.fit(X_train, y_train)

# Evaluate final model
final_metrics = final_model.evaluate(X_test, y_test)
print("\\nFinal Model Performance:")
for metric, value in final_metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# 8. Model Interpretation
print("\\nGenerating SHAP explanations...")
try:
    explainer = final_model.explain(X_test)
    # Plot SHAP summary
    import shap
    shap.summary_plot(explainer, X_test, plot_type="bar")
    plt.title("SHAP Feature Importance")
    plt.show()
except Exception as e:
    print(f"Error generating SHAP explanations: {e}")

In [ ]:
# 9. Save Model
os.makedirs("../models", exist_ok=True)
model_path = "../models/best_model.pkl"
final_model.save(model_path)
print(f"\\nModel saved to {model_path}")

# Verify model can be loaded
loaded_model = XGBoostModel.load(model_path)
print("Model loaded successfully!")